# За вкъщи: упражнение 2, част 3

**Изображения и линейна класификация**

1. Пишете кода в съответните клетки на тази тетрадка.
2. Преди предаване рестартирайте ядрото и изпълнете цялата тетрадка.
3. Изпратете я на mkorudzhiev@uni-sofia.bg с тема "Уп. 2, част 3 - <Вашето име>, <Фак. номер>".
   Изпратете заедно с нея и изображението от задача 4.

Данните са същите като в упражнението: `data/digits_png/` и `data/shapes_png/`.
Отворете тетрадката от папката на упражнение 2 и изпълнете подготовката по-долу.
Не са нужни променливи от друга тетрадка.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data/digits_png")
SHAPES_DIR = Path("data/shapes_png")
paths = sorted(DATA_DIR.rglob("*.png"))
df = pd.DataFrame({
    "filepath": [str(p) for p in paths],
    "label": [int(p.parent.name) for p in paths],
})
images = np.array([plt.imread(p) for p in df["filepath"]])
y = df["label"].values
X = images.reshape(len(images), -1)
X_train, X_test, y_train, y_test, train_images, test_images = train_test_split(
    X, y, images, test_size=0.2, random_state=42, stratify=y
)
model = Perceptron(random_state=42).fit(X_train, y_train)
predictions = model.predict(X_test)


def show_images(selected_images, titles, ncols=5):
    nrows = (len(selected_images) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.3 * ncols, 2.6 * nrows),
                             squeeze=False, layout="constrained")
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, title in zip(axes.ravel(), selected_images, titles):
        ax.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title, fontsize=10)
    plt.show()


def center_of_mass(img):
    """Средният ред и средната колона, претеглени с яркостта на пикселите."""
    total = img.sum()
    row = (img.sum(axis=1) * np.arange(img.shape[0])).sum() / total
    col = (img.sum(axis=0) * np.arange(img.shape[1])).sum() / total
    return row, col


print("За обучение:", len(y_train), "| за проверка:", len(y_test))
print("Правилно разпознати тестови изображения:", int((predictions == y_test).sum()),
      "от", len(y_test))

## Задача 1. Типични и нетипични примери

Изчислете средното изображение на цифрата 2 само от тренировъчните
изображения (`train_images` и `y_train`). За всяка тренировъчна тренировъчна цифра 2
изчислете **евклидовото разстояние** до средното изображение: корен квадратен
от сумата на квадратите на разликите в 784-те пиксела. Колкото по-малко е
разстоянието, толкова по-типичен е примерът.

Покажете трите най-близки и трите най-далечни двойки, като в заглавието
запишете разстоянието. С какво се различават най-далечните от типичните?

След това изчислете средното изображение на всяка цифра и разстоянието от
всяко **тестово** изображение до средното на неговия верен клас. Сравнете
средното разстояние на правилно и на грешно разпознатите тестови изображения.
Разпознават ли се нетипичните примери по-често грешно?

<details><summary>Подсказка</summary>

`np.sqrt(((twos - mean_two) ** 2).sum(axis=(1, 2)))` дава разстоянията на
всички двойки наведнъж, а `np.argsort` подрежда позициите по разстояние.
Във втората част `class_means[y_test]` избира средното на верния клас за
всяко тестово изображение, а `predictions == y_test` отделя правилните
от грешните.

</details>

In [ ]:
# twos = ...
# mean_two = ...
# distances = ...
# order = np.argsort(distances)
# show_images(..., ..., ncols=6)
#
# class_means = np.array([... for digit in range(10)])
# test_distances = ...
# correct = predictions == y_test
# print("Правилни:", ..., "| грешни:", ...)

*Отговор за задача 1:*

## Задача 2. Огледални цифри

Обърнете огледално (ляво-дясно) всички тестови изображения и ги подайте на
обучения `model`. Колко от тях се разпознават правилно? Сравнете с
оригиналните изображения поотделно за всяка цифра.

Преди да изпълните кода, запишете кои цифри очаквате да се разпознават и
след обръщането. После сравнете очакването с резултата. Защо някои цифри,
които изглеждат симетрични, губят много от правилните си отговори?

<details><summary>Подсказка</summary>

`test_images[:, :, ::-1]` обръща колоните на всички изображения наведнъж,
също като `np.fliplr` за едно изображение. Преди `predict` подредете
изображенията в редове от 784 признака. За цифрата `digit` броят правилни
отговори е `(mirror_predictions[y_test == digit] == digit).sum()`.

</details>

In [ ]:
# mirrored = ...
# mirror_predictions = ...
# print("Огледални, правилни:", ..., "от", len(y_test))
# for digit in range(10):
#     mask = y_test == digit
#     print(digit, ..., ...)

*Отговор за задача 2:*

## Задача 3. Центриране на фигурите

В задачата от упражнението Perceptron разпознава около 70% от фигурите в
`data/shapes_png/`. Фигурите са на случайно място в изображението, а в
упражнението видяхте, че моделът е чувствителен към изместване.

Заредете фигурите и центрирайте всяко изображение по център на масата
(функцията `center_of_mass` е дадена в подготовката). Обучете нов модел
`Perceptron(random_state=42)`: върху върху центрираните
изображения, с разделяне (`test_size=0.2`, `random_state=42`,
`stratify`). Колко тестови фигури разпознава новия модел?

Защо центрирането помага много повече на фигурите, отколкото на цифрите от
MNIST? Какви разлики между фигурите остават и след центрирането?

<details><summary>Подсказка</summary>

Центрирането е като в част 2 на упражнението:
`np.roll(img, (round(13.5 - row), round(13.5 - col)), axis=(0, 1))`, където
`row, col = center_of_mass(img)`. Етикетът на фигурата е името на папката.

</details>

In [ ]:
# shape_paths = sorted(SHAPES_DIR.rglob("*.png"))
# shape_images = ...
# shape_labels = ...
# centered_images = np.array([... for img in shape_images])
# for name, imgs in [("оригинални", shape_images), ("центрирани", centered_images)]:
#     ...
#     print(name, ...)

*Отговор за задача 3:*

## Задача 4. Собствена цифра

Напишете цифра по ваш избор с дебел флумастер върху бял лист и я снимайте,
или я нарисувайте в Paint. Запишете файла до тетрадката, например като
`моя_цифра.png`.

Подгответе изображението така, както е подготвен MNIST:
1. заредете го с Pillow като сиво изображение и нормализирайте стойностите
   до интервала от 0 до 1;
2. обърнете го, защото на снимката цифрата е тъмна върху светъл фон;
3. изрежете цифрата по рамката ѝ и я намалете така, че по-дългата ѝ страна
   да е 20 пиксела;
4. поставете я в празно изображение 28x28 и я центрирайте по център на масата.

Покажете снимката и подготвеното изображение. Направете предсказание и
покажете десетте линейни оценки (`model.decision_function`). Не е
необходимо моделът да разпознае цифрата правилно. Важни са подготовката и
обяснението: с какво вашето изображение се различава от примерите в MNIST?

<details><summary>Подсказка</summary>

`Image.open(path).convert("L")` зарежда сиво изображение. След обръщането
сенките от листа стават малки положителни стойности. Занулете ги с праг,
например `digit[digit < 0.3] = 0`. За намаляването използвайте
`Image.fromarray((cropped * 255).astype(np.uint8)).resize((ширина, височина))`
и разделете резултата отново на 255. Ако моделът сбърка, сравнете
подготвеното изображение с няколко примера от същия клас.

</details>

In [ ]:
# intended_digit = ...
# photo = Image.open("моя_цифра.png").convert("L")
# digit = ...
# cropped = ...
# small = ...
# prepared = np.zeros((28, 28))
# ...
# print("Замислена:", intended_digit, "| предсказание:", ...)
# print("Оценки:", ...)

*Отговор за задача 4:*